In [1]:
# 7-9-2026

In [2]:
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix
from scipy.stats import spearmanr, kendalltau

In [3]:
# load embeddings, set domain_id as index
df_embeddings = pd.read_csv("domain_embeddings_v3.csv")
df_embeddings.set_index("domain_id", inplace=True)

In [4]:
df_embeddings.head()

,e_0,e_1,e_2,e_3,e_4,e_5,e_6,e_7,e_8
domain_id,,,,,,,,,
0,-0.153192,0.178042,-0.220264,0.007731,-0.307213,0.146843,0.196149,0.265362,-0.027542
1,-0.244916,0.217204,-0.265037,0.471137,-0.148067,1.016517,0.633035,0.260105,-0.106379
2,0.008171,0.215184,-0.153718,-0.231967,-0.165269,-0.026749,-0.137573,0.131834,1.081505
4,-0.112366,-0.011225,0.986331,-0.142931,-0.113171,-0.249637,0.218600,0.164212,-0.046884
5,-0.218327,0.145313,-0.318968,-0.022520,-0.284123,0.194040,-0.039714,0.238028,0.039042


In [5]:
# raw embedding distance matrix, no normalization
dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
# no scaling because the fact that some nodes have greater scales than others actually means something for downstream prediction
# nodes having differnt scales actually means something to the network

C:\Users\Yash\AppData\Local\Temp\ipykernel_5268\2764295889.py:2: DeprecationWarning: `distance_matrix` is deprecated in favor of `scipy.spatial.distance.cdist` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
C:\Users\Yash\AppData\Local\Temp\ipykernel_5268\2764295889.py:2: DeprecationWarning: `minkowski_distance` is deprecated in favor of `scipy.spatial.distance.minkowski` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
C:\Users\Yash\AppData\Local\Temp\ipykernel_5268\2764295889.py:2: DeprecationWarning: `minkowski_distance_p` is deprecated in favor of `scipy.spatial.distance.minkowski` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default


In [6]:
# negate so bigger=lower transferability. same  as rawdist
dist_raw_df = pd.DataFrame(-dist_raw, index=df_embeddings.index, columns=df_embeddings.index)

In [7]:
dist_raw_df.head()

domain_id,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
domain_id,,,,,,,,,,,,,,,,,,,,,
0,-0.000000,-1.097939,-1.224242,-1.312257,-0.282035,-1.017579,-0.467399,-0.500967,-0.743199,-0.986187,...,-0.499771,-0.663056,-0.637163,-0.755550,-0.284533,-1.695991,-1.189250,-1.618741,-0.711346,-0.870055
1,-1.097939,-0.000000,-1.918557,-1.949771,-1.192345,-1.673653,-1.355785,-0.734422,-1.614340,-1.711187,...,-1.265848,-1.444148,-1.484107,-1.116822,-1.237794,-1.797731,-1.856895,-2.274187,-1.513679,-1.218239
2,-1.224242,-1.918557,-0.000000,-1.681366,-1.139217,-1.505432,-1.248529,-1.477491,-0.825922,-1.483942,...,-1.337047,-1.256540,-1.203518,-1.223187,-1.228883,-2.326971,-1.225624,-1.673217,-1.225854,-1.679968
4,-1.312257,-1.949771,-1.681366,-0.000000,-1.435155,-0.390485,-1.160814,-1.404474,-1.499753,-1.857015,...,-1.563368,-0.950193,-0.767696,-1.652259,-1.183470,-2.497576,-2.054139,-0.640578,-0.686520,-1.290135
5,-0.282035,-1.192345,-1.139217,-1.435155,-0.000000,-1.156522,-0.628730,-0.667764,-0.538669,-0.898016,...,-0.595351,-0.756316,-0.755120,-0.593574,-0.498179,-1.795413,-0.985061,-1.656674,-0.804482,-1.137092


In [8]:
dist_raw_df.to_csv("embedding_matrix_v3.csv")

In [9]:
embedding_matrix = pd.read_csv("embedding_matrix_v3.csv")
embedding_matrix.set_index("domain_id", inplace=True)
embedding_matrix.index.name = None
embedding_matrix.index = embedding_matrix.index.astype(int)
embedding_matrix.columns = embedding_matrix.columns.astype(int)
embedding_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-1.097939,-1.224242,-1.312257,-0.282035,-1.017579,-0.467399,-0.500967,-0.743199,-0.986187,...,-0.499771,-0.663056,-0.637163,-0.755550,-0.284533,-1.695991,-1.189250,-1.618741,-0.711346,-0.870055
1,-1.097939,-0.000000,-1.918557,-1.949771,-1.192345,-1.673653,-1.355785,-0.734422,-1.614340,-1.711187,...,-1.265848,-1.444148,-1.484107,-1.116822,-1.237794,-1.797731,-1.856895,-2.274187,-1.513679,-1.218239
2,-1.224242,-1.918557,-0.000000,-1.681366,-1.139217,-1.505432,-1.248529,-1.477491,-0.825922,-1.483942,...,-1.337047,-1.256540,-1.203518,-1.223187,-1.228883,-2.326971,-1.225624,-1.673217,-1.225854,-1.679968
4,-1.312257,-1.949771,-1.681366,-0.000000,-1.435155,-0.390485,-1.160814,-1.404474,-1.499753,-1.857015,...,-1.563368,-0.950193,-0.767696,-1.652259,-1.183470,-2.497576,-2.054139,-0.640578,-0.686520,-1.290135
5,-0.282035,-1.192345,-1.139217,-1.435155,-0.000000,-1.156522,-0.628730,-0.667764,-0.538669,-0.898016,...,-0.595351,-0.756316,-0.755120,-0.593574,-0.498179,-1.795413,-0.985061,-1.656674,-0.804482,-1.137092
6,-1.017579,-1.673653,-1.505432,-0.390485,-1.156522,-0.000000,-0.915974,-1.072001,-1.277358,-1.683384,...,-1.325665,-0.798268,-0.543303,-1.394145,-0.909483,-2.328337,-1.856650,-0.905603,-0.446244,-1.048739
7,-0.467399,-1.355785,-1.248529,-1.160814,-0.628730,-0.915974,-0.000000,-0.748840,-0.930197,-1.035489,...,-0.571276,-0.400419,-0.436938,-1.092997,-0.236736,-1.741277,-1.425664,-1.422897,-0.661609,-0.768973
8,-0.500967,-0.734422,-1.477491,-1.404474,-0.667764,-1.072001,-0.748840,-0.000000,-1.107048,-1.376819,...,-0.833881,-0.891808,-0.851638,-0.840089,-0.610816,-1.811743,-1.548772,-1.767444,-0.891229,-0.801867
11,-0.743199,-1.614340,-0.825922,-1.499753,-0.538669,-1.277358,-0.930197,-1.107048,-0.000000,-1.063199,...,-0.928576,-0.946118,-0.917515,-0.667673,-0.847001,-2.162454,-0.811292,-1.579275,-0.917733,-1.531329
12,-0.986187,-1.711187,-1.483942,-1.857015,-0.898016,-1.683384,-1.035489,-1.376819,-1.063199,-0.000000,...,-0.800625,-1.170714,-1.262278,-1.302468,-1.069928,-1.391386,-0.845381,-1.978852,-1.403774,-1.506500


In [10]:
rawdist_matrix = pd.read_csv("../baselines/rawdist_matrix.csv")
rawdist_matrix.set_index("Unnamed: 0", inplace=True)
rawdist_matrix.index.name = None
rawdist_matrix.index = rawdist_matrix.index.astype(int)
rawdist_matrix.columns = rawdist_matrix.columns.astype(int)
rawdist_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-6.672423,-11.045133,-7.986460,-3.308284,-7.958821,-6.068376,-4.666705,-5.184888,-4.589527,...,-7.662378,-6.407206,-8.197987,-5.296501,-4.204545,-11.505974,-7.273316,-13.276326,-6.751253,-3.125536
1,-6.672423,-0.000000,-13.599319,-9.841238,-5.547623,-9.436429,-9.730264,-5.698760,-8.974480,-8.392720,...,-11.038897,-10.159206,-10.744922,-5.711799,-7.998925,-13.061897,-11.181847,-15.567050,-8.537365,-6.931799
2,-11.045133,-13.599319,-0.000000,-13.082436,-11.499412,-12.708249,-9.285989,-13.684210,-9.565427,-8.687225,...,-7.894825,-9.563264,-10.696453,-11.590126,-9.063832,-9.620238,-7.577379,-16.658596,-9.353973,-11.288658
4,-7.986460,-9.841238,-13.082436,-0.000000,-8.019986,-3.351193,-7.878595,-9.379059,-9.467587,-9.484902,...,-10.290864,-7.254905,-6.459036,-9.505884,-7.903673,-13.860378,-11.642435,-9.120748,-5.452087,-7.889157
5,-3.308284,-5.547623,-11.499412,-8.019986,-0.000000,-7.551036,-6.591061,-4.260023,-4.860621,-5.079144,...,-8.673302,-7.712744,-8.004912,-4.035172,-5.374218,-12.376469,-7.461498,-12.675088,-6.924344,-4.980700
6,-7.958821,-9.436429,-12.708249,-3.351193,-7.551036,-0.000000,-7.307292,-8.895403,-9.258151,-9.329919,...,-9.975772,-7.488843,-4.461359,-9.152761,-7.552179,-13.639737,-11.240952,-8.347249,-5.771723,-7.863910
7,-6.068376,-9.730264,-9.285989,-7.878595,-6.591061,-7.307292,-0.000000,-8.832793,-7.072502,-5.136909,...,-4.850997,-4.276811,-5.501850,-8.444666,-2.986468,-8.724586,-7.209533,-12.837734,-5.064828,-5.701705
8,-4.666705,-5.698760,-13.684210,-9.379059,-4.260023,-8.895403,-8.832793,-0.000000,-7.504786,-7.592999,...,-10.594288,-8.880334,-9.930977,-5.531485,-7.389167,-14.100157,-10.431665,-13.559093,-9.241698,-5.032279
11,-5.184888,-8.974480,-9.565427,-9.467587,-4.860621,-9.258151,-7.072502,-7.504786,-0.000000,-5.607953,...,-8.563762,-8.084519,-8.778158,-4.434500,-6.187654,-12.536701,-4.309561,-13.146039,-7.716434,-6.559332
12,-4.589527,-8.392720,-8.687225,-9.484902,-5.079144,-9.329919,-5.136909,-7.592999,-5.607953,-0.000000,...,-6.457259,-6.402053,-8.537121,-6.857717,-4.288140,-9.152598,-5.360677,-14.267618,-6.997054,-5.922538


In [11]:
true_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")
true_matrix.set_index("Unnamed: 0", inplace=True)
true_matrix.index.name = None
true_matrix.index = true_matrix.index.astype(int)
true_matrix.columns = true_matrix.columns.astype(int)
true_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770
6,0.178282,0.238235,0.034447,0.220869,0.191487,0.317381,0.118250,0.163631,0.119401,0.021430,...,0.202794,0.247076,0.125801,0.255555,0.201974,-0.068777,0.099243,0.321076,0.102840,-0.017236
7,0.113396,0.054282,0.061550,0.069370,0.044484,0.109329,0.329688,0.092023,0.159883,0.051387,...,0.184071,0.179713,0.142222,0.192045,0.209176,0.035736,0.042035,0.159817,0.008022,0.022467
8,0.181002,0.228547,-0.006209,0.023346,0.133774,0.117255,0.194411,0.424899,0.064289,-0.004919,...,0.093679,0.136592,0.101116,0.173933,0.095452,-0.003312,0.050942,0.240088,0.051530,0.007182
11,0.214508,0.227264,0.109767,0.124790,0.237549,0.076848,0.107019,0.207917,0.551275,-0.003236,...,0.256072,0.166287,0.102690,0.366209,0.181682,-0.007782,0.147501,0.206234,-0.057620,0.064859
12,0.118927,0.175802,-0.016922,0.112255,0.159920,0.070012,0.077276,0.026258,0.103184,0.502334,...,0.163311,-0.091433,0.060118,0.163172,0.119053,0.080449,0.089125,0.118189,0.068853,-0.070408


In [12]:
full_tau, _ = kendalltau(embedding_matrix.values.flatten(), true_matrix.values.flatten())
full_tau

np.float64(0.25158735130349574)

In [13]:
full_s, _ = spearmanr(embedding_matrix.values.flatten(), true_matrix.values.flatten())
full_s

np.float64(0.3638956934761521)

In [14]:
full_s, _ = spearmanr(embedding_matrix.values.flatten(), rawdist_matrix.values.flatten())
full_s

np.float64(0.7013948537828603)

In [15]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], true_matrix.values[mask])
print(f"off diag spearman: {off_diag_tau:.4f}")

off diag spearman: 0.2065


In [16]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(rawdist_matrix.values[mask], true_matrix.values[mask])
print(f"off diag spearman: {off_diag_tau:.4f}")

off diag spearman: 0.0943


In [17]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], true_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")

off diag kendal tau: 0.2065


In [18]:
full_tau, _ = kendalltau(embedding_matrix.values.flatten(), rawdist_matrix.values.flatten())
full_tau

np.float64(0.5119825708061002)

In [19]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], rawdist_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")

off diag kendal tau: 0.4824


In [20]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(true_matrix.values[mask], embedding_matrix.values[mask])
print(f"off diag sparman: {off_diag_tau:.4f}")

off diag sparman: 0.3046


In [21]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(embedding_matrix.values[mask], true_matrix.values[mask])
print(f"off diag spearman: {off_diag_tau:.4f}")

off diag spearman: 0.3046
